In [ ]:
# 1 - Imports et chargement des données

import pandas as pd
import numpy as np
from datetime import datetime

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train = pd.read_csv("reservations_train.csv", parse_dates=["date_reservation", "date_arrivee"])
test = pd.read_csv("reservations_test.csv", parse_dates=["date_reservation", "date_arrivee"])

print(train.shape, test.shape)
print(train["reservation_annulee"].value_counts(normalize=True))

(8000, 34) (2000, 33)
reservation_annulee
0    0.741625
1    0.258375
Name: proportion, dtype: float64


In [15]:
# 2 - Split temporel

train_sorted = train.sort_values("date_reservation").reset_index(drop=True)

cutoff_idx = int(len(train_sorted) * 0.8)
cutoff_date = train_sorted.iloc[cutoff_idx]["date_reservation"]

X_train_raw = train_sorted[train_sorted["date_reservation"] < cutoff_date].copy()
X_valid_raw = train_sorted[train_sorted["date_reservation"] >= cutoff_date].copy()

print("Cutoff Date : ", cutoff_date)
print("Train : ", X_train_raw.shape, "| Valid : ", X_valid_raw.shape)
print("Taux annulation train : ", X_train_raw["reservation_annulee"].mean().round(3))
print("Taux annulation valid : ", X_valid_raw["reservation_annulee"].mean().round(3))

Cutoff Date :  2024-11-28 00:00:00
Train :  (6395, 34) | Valid :  (1605, 34)
Taux annulation train :  0.256
Taux annulation valid :  0.269


In [ ]:
# 3 - Préparation des colonnes (Features de base)

target_col = "reservation_annulee"
id_col = "reservation_id"
date_cols = ["date_reservation", "date_arrivee"]

cat_cols = [
    "region_hotel", "ville", "type_destination", "hotel_id", "segment_client",
    "marche_origine", "canal_reservation", "moyen_transport", "formule_repas",
    "tarif_remboursable", "type_acompte", "client_type", "agent_id"
]

num_cols = [
    "categorie_hotel", "delai_reservation_jours", "nuits", "adultes", "enfants",
    "chambres", "prix_moyen_nuit_eur", "remise_pct", "montant_total_eur",
    "reservations_passees", "annulations_passees", "demandes_speciales",
    "modifications_reservation", "jours_liste_attente", "evenement_majeur",
    "haute_saison_regionale", "arrivee_weekend"
]

# agent_id : NaN = réservation directe, on le rend explicite plutôt que de l'imputer
for df in [X_train_raw, X_valid_raw, test]:
    df["agent_id"] = df["agent_id"].fillna("DIRECT")


In [ ]:
# 4 - Baseline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report, confusion_matrix

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])

X_train = X_train_raw[num_cols + cat_cols]
y_train = X_train_raw[target_col]
X_valid = X_valid_raw[num_cols + cat_cols]
y_valid = X_valid_raw[target_col]

baseline_model.fit(X_train, y_train)

proba_valid = baseline_model.predict_proba(X_valid)[:, 1]
pred_valid = (proba_valid >= 0.5).astype(int)

print("F1 baseline (seuil 0.5):", f1_score(y_valid, pred_valid))
print("Precision:", precision_score(y_valid, pred_valid))
print("Recall:", recall_score(y_valid, pred_valid))
print("ROC-AUC:", roc_auc_score(y_valid, proba_valid))
print(classification_report(y_valid, pred_valid))

F1 baseline (seuil 0.5): 0.4532871972318339
Precision: 0.36187845303867405
Recall: 0.6064814814814815
ROC-AUC: 0.6465654699883174
              precision    recall  f1-score   support

           0       0.81      0.61      0.69      1173
           1       0.36      0.61      0.45       432

    accuracy                           0.61      1605
   macro avg       0.58      0.61      0.57      1605
weighted avg       0.69      0.61      0.63      1605



In [ ]:
# 5 - Feature engineering

def engineer_features(df):
    df = df.copy()

    # Features temporelles (date d'arrivée uniquement — connue au moment de la prédiction)
    df["mois_arrivee"] = df["date_arrivee"].dt.month
    df["jour_semaine_arrivee"] = df["date_arrivee"].dt.dayofweek
    df["trimestre_arrivee"] = df["date_arrivee"].dt.quarter

    # Ratio délai de réservation / durée du séjour (réservation très en avance vs dernière minute)
    df["ratio_delai_nuits"] = df["delai_reservation_jours"] / (df["nuits"] + 1)

    # Prix par personne
    df["personnes_totales"] = df["adultes"] + df["enfants"].fillna(0)
    df["prix_par_personne"] = df["montant_total_eur"] / df["personnes_totales"].replace(0, 1)

    # Historique client : taux d'annulation passé (attention: uniquement ratio, pas la cible actuelle)
    df["taux_annulation_historique"] = df["annulations_passees"] / (df["reservations_passees"] + 1)
    df["client_nouveau_sans_historique"] = (df["reservations_passees"] == 0).astype(int)

    # Réservation à risque : sans acompte ET remboursable (combinaison observée comme très corrélée)
    df["sans_engagement"] = ((df["type_acompte"] == "aucun") & (df["tarif_remboursable"] == "oui")).astype(int)

    # Réservation longtemps à l'avance (> 90 jours) — souvent plus volatile
    df["reservation_tres_anticipee"] = (df["delai_reservation_jours"] > 90).astype(int)

    return df

X_train_fe = engineer_features(X_train_raw)
X_valid_fe = engineer_features(X_valid_raw)
test_fe = engineer_features(test)

num_cols_fe = num_cols + [
    "mois_arrivee", "jour_semaine_arrivee", "trimestre_arrivee",
    "ratio_delai_nuits", "prix_par_personne", "taux_annulation_historique",
    "client_nouveau_sans_historique", "sans_engagement", "reservation_tres_anticipee"
]

In [ ]:
# 6 - Modèles alternatifs

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

preprocessor_fe = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols_fe),
    ("cat", categorical_transformer, cat_cols)
], sparse_threshold=0
)

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("classifier", RandomForestClassifier(
        n_estimators=400, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
])

hgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("classifier", HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.05,
        random_state=RANDOM_STATE
    ))
])

X_train_full = X_train_fe[num_cols_fe + cat_cols]
X_valid_full = X_valid_fe[num_cols_fe + cat_cols]

results = {}
for name, model in [("RandomForest", rf_model), ("HistGradientBoosting", hgb_model)]:
    model.fit(X_train_full, y_train)
    proba = model.predict_proba(X_valid_full)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        "f1": f1_score(y_valid, pred),
        "precision": precision_score(y_valid, pred),
        "recall": recall_score(y_valid, pred),
        "roc_auc": roc_auc_score(y_valid, proba)
    }
    print(name, results[name])


RandomForest {'f1': 0.44663677130044843, 'precision': 0.3645680819912152, 'recall': 0.5763888888888888, 'roc_auc': 0.6490342111079537}
HistGradientBoosting {'f1': 0.1711229946524064, 'precision': 0.37209302325581395, 'recall': 0.1111111111111111, 'roc_auc': 0.6384409238735752}


In [ ]:
# 7 - Choix du seuil de décision

def best_threshold(y_true, y_proba):
    thresholds = np.arange(0.1, 0.9, 0.01)
    scores = [f1_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    best_idx = np.argmax(scores)
    return thresholds[best_idx], scores[best_idx]

# Exemple avec le modèle final retenu (remplace best_model par celui que tu gardes)
best_model = hgb_model  # ou rf_model, ou baseline_model — selon ce qui gagne
proba_valid_best = best_model.predict_proba(X_valid_full)[:, 1]
seuil_opt, f1_opt = best_threshold(y_valid, proba_valid_best)
print(f"Seuil optimal: {seuil_opt:.2f} — F1: {f1_opt:.4f}")


Seuil optimal: 0.23 — F1: 0.4705


In [ ]:
# 8 - Analyse d'erreurs

pred_valid_best = (proba_valid_best >= seuil_opt).astype(int)

errors_df = X_valid_raw.copy()
errors_df["proba"] = proba_valid_best
errors_df["pred"] = pred_valid_best
errors_df["reel"] = y_valid.values

faux_positifs = errors_df[(errors_df["pred"] == 1) & (errors_df["reel"] == 0)]
faux_negatifs = errors_df[(errors_df["pred"] == 0) & (errors_df["reel"] == 1)]

print("5 faux positifs (prédits annulés, en réalité maintenus):")
print(faux_positifs[["reservation_id","type_acompte","tarif_remboursable","canal_reservation","delai_reservation_jours","proba"]].head(5))

print("5 faux négatifs (prédits maintenus, en réalité annulés):")
print(faux_negatifs[["reservation_id","type_acompte","tarif_remboursable","canal_reservation","delai_reservation_jours","proba"]].head(5))

5 faux positifs (prédits annulés, en réalité maintenus):
     reservation_id type_acompte tarif_remboursable    canal_reservation  \
6400        R000308        aucun                oui           entreprise   
6401        R001943        aucun                oui  plateforme_en_ligne   
6404        R009259        aucun                oui  plateforme_en_ligne   
6407        R007351        aucun                oui  plateforme_en_ligne   
6414        R004025        aucun                oui            telephone   

      delai_reservation_jours     proba  
6400                       45  0.269456  
6401                       43  0.288980  
6404                       20  0.324308  
6407                       24  0.410715  
6414                       30  0.405635  
5 faux négatifs (prédits maintenus, en réalité annulés):
     reservation_id type_acompte tarif_remboursable    canal_reservation  \
6402        R006182        aucun                oui               agence   
6418        R000902      

In [28]:
# Si RandomForest ou HistGradientBoosting retenu
import matplotlib.pyplot as plt

if hasattr(best_model.named_steps["classifier"], "feature_importances_"):
    feat_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    importances = best_model.named_steps["classifier"].feature_importances_
    top_features = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(15)
    print(top_features)

In [ ]:
# 9 - Géneration de submission.csv

X_test_full = test_fe[num_cols_fe + cat_cols]
proba_test = best_model.predict_proba(X_test_full)[:, 1]
pred_test = (proba_test >= seuil_opt).astype(int)

submission = pd.DataFrame({
    "reservation_id": test["reservation_id"],
    "probabilite_annulation": proba_test,
    "reservation_annulee": pred_test
})

assert len(submission) == 2000
assert (submission["reservation_id"].values == test["reservation_id"].values).all()

submission.to_csv("submission.csv", index=False)
print(submission.head())


  reservation_id  probabilite_annulation  reservation_annulee
0        R002267                0.184571                    0
1        R002852                0.224878                    0
2        R003752                0.451170                    1
3        R009926                0.295440                    1
4        R000141                0.164314                    0
